# The Complete RAG Pipeline

**Goal**: Wire hybrid retrieval into a local LLM to build an end-to-end question-answering system over arXiv papers.

By the end of this notebook you will understand:
- How Ollama works: the `/api/generate` endpoint, model management, streaming NDJSON
- Why naive context building kills LLM performance — and the 80% prompt reduction fix
- How to build a prompt that enforces citation and grounding
- The RAG orchestrator: retrieval → context → generation in one clean class
- Server-Sent Events (SSE): how streaming works end-to-end from Ollama → FastAPI → browser
- Why you need two API endpoints: standard and streaming
- How to build a Gradio interface on top of a streaming API

---

## The Complete Pipeline

```
User question
      │
      ▼
HybridSearchService          ← what you built last phase
  (BM25 + kNN → RRF)
      │
      ▼  top-K chunks
ContextBuilder               ← NEW: strips noise, builds optimised prompt
  (strip metadata, format, truncate)
      │
      ▼  clean prompt
OllamaService                ← NEW: local LLM inference
  (llama3.2, streaming NDJSON)
      │
      ▼  answer + sources
FastAPI /ask endpoint        ← NEW: standard + SSE streaming variants
      │
      ▼
Gradio UI                    ← NEW: interactive demo with live streaming
```

---

## Section 1: Ollama — Local LLM Inference

### Why local LLMs?

| Concern | Cloud API (OpenAI etc.) | Ollama (local) |
|---------|------------------------|----------------|
| Data privacy | Papers sent to 3rd party | Never leaves your machine |
| Cost | Per-token billing | Free after hardware |
| Rate limits | Hard API caps | None |
| Latency | Network round-trip | Local inference only |
| Model control | Provider decides | You choose the model |

For an academic research assistant, privacy is critical — researchers may be working with unpublished papers.

### The Ollama API

Ollama exposes two endpoints we care about:
- `POST /api/generate` — single-turn completion (our main endpoint)
- `POST /api/chat`     — multi-turn conversation with message history
- `GET  /api/tags`     — list available models
- `POST /api/pull`     — download a model

With `stream: true`, `/api/generate` returns newline-delimited JSON (NDJSON) — one JSON object per token.

In [ ]:
import requests
import json
import time

OLLAMA_URL   = "http://localhost:11434"
OLLAMA_MODEL = "llama3.2:1b"  # 3b is fast; use 8b for better quality if you have RAM

# --- Check Ollama is running and what models are available ---
def check_ollama():
    try:
        r = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
        r.raise_for_status()
        models = [m["name"] for m in r.json().get("models", [])]
        return {"status": "running", "models": models}
    except Exception as e:
        return {"status": "error", "message": str(e)}

status = check_ollama()
print(f"Ollama status: {status['status']}")
if status.get("models"):
    print(f"Available models: {status['models']}")
else:
    print("No models downloaded yet.")
    print(f"Run: docker exec rag-ollama ollama pull {OLLAMA_MODEL}")

Ollama status: running
Available models: ['qwen3:8b']


In [4]:
# Pull the model if not already available
# qwen3:8b = ~2GB download. Run once, cached forever.

def ensure_model(model: str):
    """Pull a model if it's not already downloaded."""
    tags = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5).json()
    existing = [m["name"] for m in tags.get("models", [])]

    if any(model in m for m in existing):
        print(f"✅ Model '{model}' already available")
        return

    print(f"Pulling '{model}'... (this downloads ~2GB, takes a few minutes)")
    r = requests.post(
        f"{OLLAMA_URL}/api/pull",
        json={"name": model, "stream": True},
        stream=True, timeout=600
    )
    for line in r.iter_lines():
        if line:
            data = json.loads(line)
            status = data.get("status", "")
            if "pulling" in status or "downloading" in status:
                completed = data.get("completed", 0)
                total     = data.get("total", 1)
                pct = int(completed / total * 100) if total > 0 else 0
                print(f"  {status}: {pct}%", end="\r")
            elif data.get("status") == "success":
                print(f"\n✅ Model '{model}' pulled successfully")

ensure_model(OLLAMA_MODEL)

Pulling 'qwen3:8b'... (this downloads ~2GB, takes a few minutes)
  pulling 05a61d37b084: 100%
✅ Model 'qwen3:8b' pulled successfully


In [7]:
# Basic generation — non-streaming (stream=False waits for complete response)

def generate(prompt: str, model: str = OLLAMA_MODEL, stream: bool = False) -> str:
    """
    Call Ollama's /api/generate endpoint.

    With stream=False: waits for the full response, returns it as a string.
    With stream=True: streams NDJSON tokens (see Section 3 for streaming).
    """
    r = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json={
            "model":  model,
            "prompt": prompt,
            "stream": stream,
            "options": {
                "temperature":    0.1,   # low temp = focused, factual answers
                "num_predict":    512,   # max tokens to generate
                "top_p":          0.9,
                "repeat_penalty": 1.1,
            }
        },
        timeout=120,
    )
    r.raise_for_status()
    return r.json()["response"]


# Quick sanity test — is the LLM working?
print("Testing LLM (non-streaming)...")
t0 = time.time()
response = generate("In one sentence, what is the transformer architecture?")
elapsed  = time.time() - t0
print(f"Response ({elapsed:.1f}s): {response}")


Testing LLM (non-streaming)...


HTTPError: 500 Server Error: Internal Server Error for url: http://localhost:11434/api/generate

### 💡 Key insight: `temperature=0.1` for RAG

Temperature controls randomness:
- `temperature=1.0`: creative, varied, sometimes hallucinates
- `temperature=0.1`: deterministic, factual, grounded

For RAG, you want the model to extract and summarise from context, not invent. Low temperature keeps it close to what the retrieved chunks actually say.

### 💡 Key insight: `num_predict=512` — why limit output length?

Longer outputs mean:
- More tokens to generate → slower
- More chance of drifting off-topic
- Higher memory pressure for the LLM

For a Q&A system, a focused 2–4 paragraph answer is better than a sprawling essay. 512 tokens (~400 words) is a good default.

---

## Section 2: Context Building — The 80% Prompt Reduction

### The naive approach and why it fails

Most RAG tutorials do this:
```python
context = "\n".join([str(chunk) for chunk in retrieved_chunks])
prompt  = f"Answer based on: {context}\n\nQuestion: {question}"
```

Each `chunk` object typically contains: `arxiv_id`, `chunk_id`, `chunk_text`, `section_name`, `chunk_index`, `title`, `abstract`, `authors`, `categories`, `published_at`, `score`.

Dumping all of this into the prompt means:
- The LLM wastes context window on metadata it doesn't need to *answer* the question
- More tokens = slower generation
- Noise can cause the LLM to get confused by irrelevant details

**The fix**: only include what the LLM needs:
- The chunk text (obviously)
- The paper title (for citation)
- The arXiv ID (for citation link)
- Truncate long chunk_text to a max length

This reduces prompt size by ~80% while keeping all information needed for a grounded, cited answer.

In [ ]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class ContextChunk:
    """A minimal context chunk — only what the LLM needs."""
    arxiv_id: str
    title:    str
    text:     str   # truncated chunk_text


# Simulate what comes back from HybridSearchService
SAMPLE_HITS = [
    {
        "arxiv_id":     "2301.00001",
        "chunk_id":     "uuid-1",
        "chunk_text":   "Title: Attention Is All You Need\nAbstract: We propose...\n\nSection: Introduction\n\nThe transformer model introduced by Vaswani et al. (2017) relies entirely on self-attention mechanisms, dispensing with recurrence and convolutions. This architecture achieved state-of-the-art performance on translation tasks while being significantly faster to train.",
        "section_name": "Introduction",
        "chunk_index":  0,
        "title":        "Attention Is All You Need",
        "abstract":     "We propose a new simple network architecture, the Transformer...",
        "authors":      ["Vaswani, A.", "Shazeer, N."],
        "categories":   ["cs.LG", "cs.CL"],
        "published_at": "2017-06-12",
        "score":        0.94,
    },
    {
        "arxiv_id":     "2302.00002",
        "chunk_id":     "uuid-2",
        "chunk_text":   "Title: BERT: Pre-training of Deep Bidirectional Transformers\nAbstract: We introduce BERT...\n\nSection: Methods\n\nBERT uses a bidirectional transformer encoder. Unlike previous models which read text left-to-right, BERT reads the entire sequence of words at once. This bidirectional approach allows BERT to understand context from both directions simultaneously.",
        "section_name": "Methods",
        "chunk_index":  1,
        "title":        "BERT: Pre-training of Deep Bidirectional Transformers",
        "abstract":     "We introduce a new language representation model called BERT...",
        "authors":      ["Devlin, J."],
        "categories":   ["cs.CL"],
        "published_at": "2018-10-11",
        "score":        0.87,
    },
]


def build_context(
    hits: list[dict],
    max_chunks: int = 5,
    max_chars_per_chunk: int = 800,
) -> tuple[list[ContextChunk], str]:
    """
    Build a minimal context from search hits.

    Returns:
      - List of ContextChunk (for the sources section in the response)
      - A formatted context string ready to insert into the prompt

    The 80% reduction comes from:
      1. Keeping only arxiv_id + title + text (dropping chunk_id, authors,
         categories, published_at, score, section_name, chunk_index)
      2. Truncating chunk_text to max_chars_per_chunk
      3. Stripping the context header that was added for embedding
         (the LLM doesn't need the repeated title/abstract prefix)
    """
    context_chunks = []
    context_parts  = []

    for i, hit in enumerate(hits[:max_chunks]):
        raw_text = hit.get("chunk_text", "")

        # Strip the context header that was prepended for embedding
        # The header looks like: "Title: ...\nAbstract: ...\n\nSection: ..."
        # We just want the section content
        text = _strip_embedding_header(raw_text)

        # Truncate to token budget
        if len(text) > max_chars_per_chunk:
            text = text[:max_chars_per_chunk] + "..."

        ctx = ContextChunk(
            arxiv_id = hit.get("arxiv_id", ""),
            title    = hit.get("title", "Unknown"),
            text     = text.strip(),
        )
        context_chunks.append(ctx)

        # Format for prompt: numbered, minimal
        context_parts.append(
            f"[{i+1}] Paper: \"{ctx.title}\" (arxiv:{ctx.arxiv_id})\n{ctx.text}"
        )

    context_str = "\n\n".join(context_parts)
    return context_chunks, context_str


def _strip_embedding_header(text: str) -> str:
    """Remove the 'Title: ... Abstract: ... Section: ...' header prepended for embedding."""
    # Find 'Section:' line — content after it is what we actually want
    lines = text.split("\n")
    for i, line in enumerate(lines):
        if line.startswith("Section:"):
            return "\n".join(lines[i+1:]).strip()
    # If no Section: header found, return as-is (paragraph fallback chunks)
    return text.strip()


# Demonstrate the reduction
naive_size    = sum(len(str(h)) for h in SAMPLE_HITS)
chunks, ctx   = build_context(SAMPLE_HITS)
optimised_size = len(ctx)

print(f"Naive context size:     {naive_size:,} chars")
print(f"Optimised context size: {optimised_size:,} chars")
print(f"Reduction:              {(1 - optimised_size/naive_size)*100:.0f}%")
print()
print("Optimised context:")
print(ctx)

In [ ]:
# Build the full RAG prompt

SYSTEM_PROMPT = """You are an expert AI research assistant helping researchers understand academic papers.

Your task: Answer the user's question based ONLY on the provided context from research papers.

Rules:
- Base your answer strictly on the provided context. Do not use outside knowledge.
- Cite your sources using [1], [2] etc. corresponding to the numbered papers.
- If the context does not contain enough information, say so clearly.
- Be concise and precise. Aim for 2-4 paragraphs.
- Do not repeat large chunks of text verbatim — summarise and synthesise."""


def build_prompt(question: str, context_str: str) -> str:
    """Assemble the full prompt from system instructions + context + question."""
    return f"""{SYSTEM_PROMPT}

Context from retrieved papers:
{context_str}

Question: {question}

Answer:"""


question = "How does the transformer architecture differ from RNNs?"
prompt   = build_prompt(question, ctx)

print(f"Full prompt ({len(prompt)} chars):")
print("-" * 60)
print(prompt[:600] + "...")

In [ ]:
# Run the complete RAG pipeline (non-streaming)
print(f"Question: {question}")
print("Generating answer...\n")

t0     = time.time()
answer = generate(prompt)
elapsed = time.time() - t0

print(f"Answer ({elapsed:.1f}s):")
print(answer)
print()
print("Sources:")
for i, c in enumerate(chunks):
    print(f"  [{i+1}] {c.title} — https://arxiv.org/abs/{c.arxiv_id}")

### 💡 Key insight: Why cite with `[1]`, `[2]`?

Numbering context chunks and instructing the model to cite `[1]`, `[2]` serves two purposes:

1. **Grounding**: it forces the model to refer to specific sources rather than generating from general knowledge
2. **Attribution**: the API response can include a `sources` list mapping numbers to paper titles + URLs — users can click through to the original papers

This is how production RAG systems distinguish themselves from plain chatbots.

---

## Section 3: Streaming — Why It Matters and How It Works

### The UX problem with non-streaming

A 512-token answer from llama3.2:3b takes ~15–30 seconds on CPU. With non-streaming:
- User submits question
- Spinner for 20 seconds
- Full answer appears at once

With streaming:
- User submits question
- First words appear within 1–2 seconds
- Answer streams in word by word, like watching someone type

The total time is the same, but perceived latency drops dramatically. **Time-to-first-token** is the metric that matters for UX, not time-to-complete.

### How Ollama streaming works

With `stream: true`, Ollama returns NDJSON (Newline-Delimited JSON):
```
{"model":"llama3.2","response":"The","done":false}
{"model":"llama3.2","response":" transformer","done":false}
{"model":"llama3.2","response":" architecture","done":false}
...more tokens...
{"model":"llama3.2","response":"","done":true,"total_duration":12000000}
```

FastAPI wraps this in **Server-Sent Events (SSE)**:
```
data: {"token": "The"}

data: {"token": " transformer"}

data: {"token": " architecture"}

data: [DONE]
```

In [ ]:
import sys

def stream_generate(prompt: str, model: str = OLLAMA_MODEL):
    """
    Generator that yields tokens one at a time from Ollama's streaming API.

    Each yield is a single token string (may be a word, partial word, or punctuation).
    The generator returns when Ollama sends {"done": true}.
    """
    r = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json={
            "model":  model,
            "prompt": prompt,
            "stream": True,
            "options": {"temperature": 0.1, "num_predict": 300},
        },
        stream=True,
        timeout=120,
    )
    r.raise_for_status()

    for line in r.iter_lines():
        if not line:
            continue
        data = json.loads(line)
        token = data.get("response", "")
        if token:
            yield token
        if data.get("done"):
            break


# Watch the response stream in real time
short_prompt = build_prompt(
    "What is self-attention in transformers? Answer in 3 sentences.",
    ctx
)

print("Streaming response (watch tokens appear):\n")
for token in stream_generate(short_prompt):
    print(token, end="", flush=True)  # flush=True makes it appear immediately
print("\n\n[stream complete]")

### 💡 Key insight: `flush=True` and SSE

In the notebook, `print(..., flush=True)` forces the output buffer to flush immediately so each token appears as it arrives.

In FastAPI, the equivalent is `StreamingResponse` with the `text/event-stream` content type:
```python
async def token_generator():
    for token in stream_generate(prompt):
        yield f"data: {json.dumps({'token': token})}\n\n"
    yield "data: [DONE]\n\n"

return StreamingResponse(token_generator(), media_type="text/event-stream")
```

The browser's EventSource API reads this and fires events as they arrive — the UI can append each token to the display without waiting for the full response.

---

## Section 4: The Full RAG Pipeline End-to-End

In [ ]:
from dataclasses import dataclass, field
from typing import Generator

@dataclass
class RAGResponse:
    """Complete response from the RAG pipeline."""
    question:    str
    answer:      str
    sources:     list[dict]   # [{arxiv_id, title, url}, ...]
    search_mode: str           # 'hybrid' or 'bm25'
    n_chunks:    int
    took_ms:     int


class RAGPipeline:
    """
    Orchestrates the full retrieve → build_context → generate pipeline.

    Wires together:
    - HybridSearchService (retrieval)
    - ContextBuilder (prompt optimisation)
    - OllamaService (LLM generation)

    Two modes:
    - ask(): returns a complete RAGResponse (waits for full answer)
    - ask_stream(): yields (token, sources) tuples for streaming
    """

    def __init__(
        self,
        search_service,   # HybridSearchService
        ollama_url:  str  = OLLAMA_URL,
        model:       str  = OLLAMA_MODEL,
        max_chunks:  int  = 5,
        max_chars_per_chunk: int = 800,
    ):
        self._search  = search_service
        self._url     = ollama_url
        self._model   = model
        self._max_chunks = max_chunks
        self._max_chars  = max_chars_per_chunk

    def ask(
        self,
        question:    str,
        use_hybrid:  bool = True,
        categories:  list[str] | None = None,
    ) -> RAGResponse:
        """Complete (non-streaming) RAG pipeline."""
        t0 = time.time()

        # Step 1: Retrieve
        search_result = self._search.search(
            query=question,
            use_hybrid=use_hybrid,
            categories=categories,
            page_size=self._max_chunks,
        )

        # Convert to dicts for context builder
        hits = [
            {
                "arxiv_id":   h.arxiv_id,
                "title":      h.title,
                "chunk_text": h.chunk_text,
            }
            for h in search_result.hits
        ]

        if not hits:
            return RAGResponse(
                question=question,
                answer="I could not find any relevant papers to answer this question.",
                sources=[],
                search_mode=search_result.search_mode,
                n_chunks=0,
                took_ms=int((time.time()-t0)*1000),
            )

        # Step 2: Build context
        chunks, context_str = build_context(hits, self._max_chunks, self._max_chars)
        prompt = build_prompt(question, context_str)

        # Step 3: Generate
        answer = generate(prompt, self._model)

        # Step 4: Format sources
        sources = [
            {
                "index":    i + 1,
                "arxiv_id": c.arxiv_id,
                "title":    c.title,
                "url":      f"https://arxiv.org/abs/{c.arxiv_id}",
            }
            for i, c in enumerate(chunks)
        ]

        return RAGResponse(
            question    = question,
            answer      = answer,
            sources     = sources,
            search_mode = search_result.search_mode,
            n_chunks    = len(chunks),
            took_ms     = int((time.time()-t0)*1000),
        )

    def ask_stream(
        self,
        question:   str,
        use_hybrid: bool = True,
        categories: list[str] | None = None,
    ) -> Generator:
        """
        Streaming RAG pipeline.

        Yields:
          - ("token", token_string) — for each LLM token
          - ("sources", sources_list) — once, after all tokens
          - ("done", None) — signal to the consumer

        The FastAPI route wraps these into SSE events.
        """
        # Retrieve + build context synchronously (fast)
        search_result = self._search.search(
            query=question, use_hybrid=use_hybrid,
            categories=categories, page_size=self._max_chunks,
        )
        hits = [
            {"arxiv_id": h.arxiv_id, "title": h.title, "chunk_text": h.chunk_text}
            for h in search_result.hits
        ]

        if not hits:
            yield ("token", "I could not find any relevant papers to answer this question.")
            yield ("sources", [])
            yield ("done", None)
            return

        chunks, context_str = build_context(hits, self._max_chunks, self._max_chars)
        prompt = build_prompt(question, context_str)

        # Stream tokens
        for token in stream_generate(prompt, self._model):
            yield ("token", token)

        # Send sources after all tokens
        sources = [
            {"index": i+1, "arxiv_id": c.arxiv_id, "title": c.title,
             "url": f"https://arxiv.org/abs/{c.arxiv_id}"}
            for i, c in enumerate(chunks)
        ]
        yield ("sources", sources)
        yield ("done", None)


print("RAGPipeline defined. Instantiate with a real HybridSearchService to run.")
print("\nIn production, create via:")
print("  from src.services.rag.factory import make_rag_pipeline")
print("  pipeline = make_rag_pipeline()")
print("  response = pipeline.ask('What is RLHF?')")

In [ ]:
# Test the streaming interface with our simulated hits
# (using local build_context + stream_generate, no search service needed)

question = "How does the transformer architecture handle sequential data differently from RNNs?"
chunks_ctx, context_str = build_context(SAMPLE_HITS)
prompt = build_prompt(question, context_str)

print(f"Question: {question}")
print(f"Context: {len(context_str)} chars from {len(chunks_ctx)} chunks")
print(f"Prompt total: {len(prompt)} chars")
print("\nStreaming answer:\n")

full_answer = []
for token in stream_generate(prompt):
    print(token, end="", flush=True)
    full_answer.append(token)

print("\n\nSources:")
for i, c in enumerate(chunks_ctx):
    print(f"  [{i+1}] {c.title} — https://arxiv.org/abs/{c.arxiv_id}")

---

## Section 5: The Dual API Design

### Why two endpoints?

| Endpoint | Method | Use case |
|----------|--------|----------|
| `POST /api/v1/ask` | Standard JSON | Programmatic API clients, batch processing |
| `GET /api/v1/ask/stream` | SSE | Browser UIs, real-time display |

Note that the streaming endpoint uses `GET` (not POST) because:
- SSE connections are initiated by the browser with `EventSource(url)` — which only supports GET
- Query parameters carry the question and options

The standard endpoint uses `POST` with a JSON body (cleaner for complex filter objects).

### SSE format

```
Content-Type: text/event-stream

data: {"type": "token", "content": "The"}

data: {"type": "token", "content": " transformer"}

data: {"type": "sources", "content": [{"index": 1, "arxiv_id": "...", "title": "...", "url": "..."}]}

data: [DONE]

```

Two blank lines separate each event (SSE spec requirement).

---

## Section 6: Gradio Interface

### Why Gradio?

Gradio turns a Python function into a web UI in ~20 lines. Perfect for:
- Quickly demoing the RAG system to others
- Interactive testing during development
- Portfolio demos (runs locally, shareable via Gradio's tunnel)

It integrates cleanly with streaming — just `yield` tokens from the predict function.

The `gradio_launcher.py` file is a standalone entry point separate from FastAPI, on port 7861.

In [ ]:
# Demonstrate Gradio streaming pattern (run separately with gradio_launcher.py)
# This shows the pattern — the actual launcher creates a real interface

import gradio as gr

def rag_predict(question: str, use_hybrid: bool, category_filter: str) -> gr.update:
    """
    Gradio predict function — yields tokens for live streaming in the UI.

    Gradio's streaming works by yielding partial results.
    Each yield updates the displayed text.
    """
    if not question.strip():
        yield "Please enter a question."
        return

    # In the real launcher, this calls pipeline.ask_stream()
    # Here we demo with a simulated stream
    categories = [category_filter] if category_filter else None
    full_response = ""

    # Simulate streaming
    chunks_ctx, context_str = build_context(SAMPLE_HITS)
    prompt = build_prompt(question, context_str)

    for token in stream_generate(prompt):
        full_response += token
        yield full_response  # Gradio replaces the text box content each yield

    # Append sources at the end
    sources_text = "\n\n---\n**Sources:**\n"
    for i, c in enumerate(chunks_ctx):
        sources_text += f"[{i+1}] [{c.title}](https://arxiv.org/abs/{c.arxiv_id})\n"
    yield full_response + sources_text


print("Gradio interface pattern demonstrated.")
print("Run 'uv run python gradio_launcher.py' to launch the full interactive UI.")
print("Open: http://localhost:7861")

---

## Summary — What You Built

```
✅ Ollama service     — model management, health checks, auto-pull
✅ Context builder    — 80% prompt reduction, strip embedding header, truncate
✅ Prompt template    — citation enforcement, grounding rules, temperature=0.1
✅ RAG pipeline       — retrieval → context → generation, complete + streaming
✅ Standard API       — POST /api/v1/ask → full JSON response with sources
✅ Streaming API      — GET /api/v1/ask/stream → SSE tokens + sources event
✅ Gradio UI          — interactive demo, live streaming, clickable source links
```

### Concepts mastered:
- **temperature=0.1 for RAG**: factual grounding over creativity
- **num_predict=512**: prevent runaway generation
- **Context header stripping**: the chunk had a header for embedding, not for the LLM
- **80% prompt reduction**: only pass what the LLM needs to *answer*, not all metadata
- **NDJSON streaming**: how Ollama delivers tokens
- **SSE format**: how FastAPI wraps streaming for browsers
- **Time-to-first-token**: why streaming feels faster even though total time is the same
- **Citation pattern**: `[1]`, `[2]` + sources list = verifiable, grounded answers
- **Dual API design**: standard (POST/JSON) for machines, streaming (GET/SSE) for UIs

---
*Branch: `feature/rag-pipeline`*